<a href="https://colab.research.google.com/github/tchingy/ai-portfolio/blob/main/hvac_report_summariser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the transformers library
!pip install transformers

In [ ]:
# Load the BART summarisation model from Hugging Face
# Using direct model loading to avoid version compatibility issue

from transformers import BartForConditionalGeneration, BartTokenizer

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print("Model loaded and ready!")

In [ ]:
# ============================================
# HVAC REPORT SUMMARISER — Task 5
# ============================================
# This function takes a long HVAC report and returns a short summary
# using the BART model loaded above

import torch

def summarise_report(report_text, max_length=60, min_length=20):
    # Step 1: Convert the text into numbers the model understands
    inputs = tokenizer(report_text, return_tensors="pt",
                       max_length=1024, truncation=True)

    # Step 2: Run the model to generate a summary
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True,
        forced_bos_token_id=0
    )

    # Step 3: Convert the numbers back into readable text
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# ---- TEST REPORT 1 ----
report_1 = """
Annual HVAC maintenance inspection carried out on 14th March 2025 at
Greenfield Office Complex, Unit 4B. The rooftop air handling unit was
inspected and found to have a blocked evaporator coil due to dust
accumulation over the past 12 months. Filter replacement was carried out
using grade F7 filters. Refrigerant levels were checked and found to be
within acceptable parameters at 410A. The condenser fan motor showed signs
of bearing wear and was flagged for replacement within the next 30 days to
prevent unit failure. Belt tension was adjusted on the supply air fan.
Control panel readings were logged and all sensors were found to be
operating within normal range. A follow-up visit is recommended in 30 days
to replace the condenser fan motor and verify system performance post
filter change.
"""

print("REPORT 1 — Full text length:", len(report_1), "characters")
print()
print("SUMMARY:")
print(summarise_report(report_1))
print()
print("=" * 60)

In [ ]:
# ---- TEST REPORT 2 ----
# Increasing max_length to 80 to avoid cutting off mid-sentence

report_2 = """
Emergency call-out attended on 22nd March 2025 at Riverside Medical Centre,
Ground Floor. The split air conditioning unit in the pharmacy storage room
failed overnight causing the room temperature to rise to 28 degrees Celsius.
Medication storage requires a maximum of 25 degrees at all times. Upon
inspection the unit was found to have a failed capacitor on the outdoor
compressor unit. The capacitor was replaced with a compatible 45uf 450v
part carried on the service vehicle. The unit was restarted and room
temperature returned to 19 degrees Celsius within 40 minutes. A full
system test was carried out and all parameters were found to be normal.
Recommended that a spare capacitor is kept on site given the age of the
unit which is now 9 years old.
"""

print("REPORT 2 — Full text length:", len(report_2), "characters")
print()
print("SUMMARY (max_length=80):")
print(summarise_report(report_2, max_length=80, min_length=30))
print()
print("=" * 60)

# ---- TEST REPORT 3 ----
report_3 = """
Planned preventative maintenance visit completed on 28th March 2025 at
Sunfield Primary School, Main Building. Four fan coil units across the
first floor classrooms were serviced. All filters were found to be heavily
soiled and were replaced with new grade G4 filters. One fan coil unit in
classroom 1C was found to have a water leak at the condensate drain
connection. The drain pipe was resealed using waterproof HVAC sealant and
tested with no further leakage observed. Thermostat calibration was checked
on all four units and one unit in classroom 1A was found to be reading
2 degrees higher than actual room temperature. The thermostat sensor was
recalibrated and verified against a calibrated reference thermometer.
All units were left in full working order. Next scheduled visit due in
6 months.
"""

print("REPORT 3 — Full text length:", len(report_3), "characters")
print()
print("SUMMARY (max_length=100):")
print(summarise_report(report_3, max_length=100, min_length=40))
print()
print("=" * 60)
print()
print("All 3 reports summarised successfully!")

In [ ]:
# ============================================
# TASK 5 — FINDINGS AND OBSERVATIONS
# ============================================

# MODEL USED: facebook/bart-large-cnn
# A summarisation model trained on CNN/Daily Mail news articles.
# Applied here to HVAC maintenance reports.

# WHAT WORKED WELL:
# - Consistently compressed reports by ~67% while keeping core findings
# - Retained technical terms (410A refrigerant, 45uf 450v, G4 filters)
# - Correctly identified the most critical issue in each report
# - Complete sentences achieved when max_length was set high enough

# PARAMETER ENGINEERING OBSERVATIONS:
# - max_length=60 caused cut-off mid-sentence (Report 1)
# - max_length=80 produced clean complete summaries (Report 2)
# - max_length=100 still cut off a longer report (Report 3)
# - Recommendation: set max_length dynamically at ~15% of input length

# LIMITATIONS IDENTIFIED:
# - Model dropped safety-critical follow-up recommendation in Report 2
#   (spare capacitor for medical facility) — it has no concept of risk priority
# - Thermostat calibration fault missed in Report 3
# - Root cause: model trained on news articles, not HVAC service data
# - Fix in production: fine-tune on labelled HVAC reports with
#   risk-weighted annotations

# BUSINESS VALUE:
# - A facilities manager reviewing 20 reports/day would save significant time
# - Could be integrated into CAFM (Computer Aided Facilities Management) systems
# - Potential to flag urgent findings automatically with additional classification layer